### Importações para Parent Document RAG

Bibliotecas novas em relação ao RAG simples:

- **ParentDocumentRetriever**: O componente central deste notebook — recupera chunks pequenos mas retorna os documentos "pai" maiores ao LLM
- **InMemoryStore**: Armazena os documentos pai em memória RAM (sem persistência em disco)
- **RunnablePassthrough**: Passa o valor de entrada sem transformação na chain LCEL
- **RunnableParallel**: Executa múltiplos runnables em paralelo com a mesma entrada
- **StrOutputParser**: Converte a saída do LLM (objeto `AIMessage`) em string simples

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [17]:
from dotenv import load_dotenv

load_dotenv()

True

In [18]:
import pandas as pd
from IPython.display import display

### Modelos de Embedding e LLM

- `OpenAIEmbeddings(model='text-embedding-3-small')`: Modelo explicitamente especificado (mais eficiente e barato)
- `max_tokens=500`: Resposta moderada — maior que o RAG simples (200), menor que o code review (1000)

In [19]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

llm = ChatOpenAI(model_name='gpt-3.5-turbo', max_tokens=500)

### Carregamento do Documento PDF

- `PyPDFLoader`: Lê o arquivo PDF e extrai o texto página por página
- `extract_images=False`: Ignora imagens, processa apenas texto
- `load_and_split()`: Carrega e já divide por páginas — cada página vira um documento separado
- Resultado: lista `pages` com 31 documentos (um por página do PDF)

In [20]:
pdf_link = './assets/anexo-projeto-de-lei.pdf'

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()

In [21]:
len(pages)

31

### Estratégia Parent-Child: Dois Splitters

Esta é a principal diferença do **Parent Document RAG** em relação ao RAG simples.

**Problema do RAG simples**: Há um trade-off difícil de resolver:
- Chunks **pequenos** → Embeddings mais precisos na busca, mas pouco contexto para o LLM responder
- Chunks **grandes** → Mais contexto para o LLM, mas embeddings menos precisos (ruído semântico)

**Solução — dois splitters com papéis distintos:**

- **child_splitter** (`chunk_size=200`): Cria chunks pequenos (~2-3 frases)
  - Usados apenas para gerar embeddings e fazer a busca
  - Não são enviados ao LLM diretamente
  - Alta precisão semântica na recuperação

- **parent_splitter** (`chunk_size=4000`, `chunk_overlap=200`): Cria chunks grandes
  - São os documentos "pai" armazenados no `InMemoryStore`
  - Enviados ao LLM como contexto após a busca
  - Garantem contexto rico para geração da resposta

**Fluxo**: Busca por chunks filhos → Encontra pai correspondente → Envia pai ao LLM

In [22]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4_000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True
)


### Dois Storages com Propósitos Diferentes

**InMemoryStore** (docstore):
- Armazena os documentos **pai** (chunks grandes de 4000 chars)
- Funciona como um dicionário em memória RAM: `{doc_id: documento}`
- Não é pesquisável por similaridade — serve apenas para recuperar pelo ID
- Não persiste entre execuções do notebook

**Chroma** (vectorstore):
- Armazena os embeddings dos chunks **filhos** (chunks pequenos de 200 chars)
- `persist_directory='childVectorDB'`: Persiste em disco (apenas os filhos/embeddings)
- Permite busca por similaridade semântica
- Cada chunk filho guarda uma referência ao ID do documento pai correspondente

In [23]:
store = InMemoryStore()

vectorstore = Chroma(embedding_function=embeddings, persist_directory='childVectorDB')

### ParentDocumentRetriever: O Coração do Parent RAG

Conecta os dois storages e implementa a estratégia parent-child:

- **vectorstore**: Onde busca (chunks filhos com embeddings)
- **docstore**: De onde recupera (chunks pais com conteúdo completo)
- **child_splitter**: Como dividir para indexar (pequeno, para embeddings precisos)
- **parent_splitter**: Como dividir para armazenar (grande, para contexto rico)

**Na busca**, o retriever faz internamente:
1. Converte a pergunta em embedding
2. Busca chunks **filhos** similares no Chroma
3. Lê o `doc_id` do pai referenciado em cada chunk filho
4. Busca os documentos **pais** correspondentes no `InMemoryStore`
5. Retorna os pais (não os filhos!) como contexto para o LLM

In [24]:
parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

In [25]:
parent_document_retriever.add_documents(pages, ids=None)

### Inspecionando o Vector Store (Chunks Filhos)

O DataFrame a seguir exibe os **2373 chunks filhos** armazenados no Chroma — são os chunks pequenos (200 chars) usados na busca por similaridade.

Note que o vector store contém apenas os filhos, não os pais. Os documentos pais (4000 chars) ficam no `InMemoryStore` e são referenciados pelo `doc_id` nos metadados de cada chunk filho.

In [26]:
data = parent_document_retriever.vectorstore.get()

df = pd.DataFrame({
    "id": data["ids"],
    "document": data["documents"],
    "metadata": data["metadatas"],
})

display(df)

,id,document,metadata
0,e22a8e74-9e7e-450e-ae47-06f9f79e896d,"PROJETO DE LEI Nº , DE 2023 \nDispõe sob...","{'creationdate': '2023-05-03T18:22:00+00:00', ..."
1,bd7c5e47-e27f-4037-b8fe-3a95275f274b,Dispõe sobre o uso da Inteligência Artificial....,{'doc_id': 'a87df713-7a1b-4f30-bcfc-c88c0ce639...
2,9e0e7933-9649-44c5-ae03-50c653a9289b,CAPÍTULO I \nDISPOSIÇÕES PRELIMINARES \nArt. 1...,{'source': './assets/anexo-projeto-de-lei.pdf'...
3,a010f412-34e5-42d0-8458-2d9cdd704dd7,"o desenvolvimento, implementação e uso respons...","{'moddate': '2023-05-03T19:15:38-03:00', 'tota..."
4,01f54c7f-3440-40bf-a53e-fb1302c851e5,"inteligência artificial (IA) no Brasil, com o ...","{'page': 0, 'producer': 'Aspose.Words for Java..."
...,...,...,...
2368,68d6a860-407c-42a9-8e15-da714a5264aa,máquina e o desenvolvimento de sistema de inte...,"{'page_label': '31', 'total_pages': 31, 'creat..."
2369,2664b32a-b615-410c-ab52-7c2bdba95a00,"contudo, implicar em prejuízo aos titulares de...","{'total_pages': 31, 'creator': 'Microsoft Offi..."
2370,0e2e78ea-9871-447c-9ac5-1ceff1770727,desdobramentos de como a regulação pode foment...,"{'creationdate': '2023-05-03T18:22:00+00:00', ..."
2371,ddae36c4-3f9e-41a4-8967-83054b4c463e,"exposto, e cientes do desafio que a matéria re...","{'start_index': 0, 'total_pages': 31, 'produce..."


### Prompt com ChatPromptTemplate

Usa `from_template` (mais simples que `from_messages`):
- Cria um prompt de role único sem separação explícita de system/user
- `{query}`: Receberá a pergunta do usuário via `RunnablePassthrough`
- `{context}`: Receberá os documentos pais recuperados pelo `ParentDocumentRetriever`

O papel de "especialista em legislação e tecnologia" orienta o LLM a responder dentro do domínio do documento carregado (projeto de lei de IA)

In [27]:
TEMPLATE = """
Você é um especialista em legislação e tecnologia. Responda a pergunta abaixo utilizando o contexto informado.

Query:
{query}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

### Composição com LCEL (LangChain Expression Language)

Aqui é usada a sintaxe moderna do LangChain para montar o pipeline:

**RunnableParallel**:
- Executa dois runnables com a **mesma entrada** simultaneamente:
  - `'query': RunnablePassthrough()` → Passa a pergunta original sem alteração
  - `'context': parent_document_retriever` → Executa a busca e retorna os documentos pais
- Resultado: `{'query': 'pergunta do usuário', 'context': [doc_pai_1, doc_pai_2, ...]}`
- Esse dicionário preenche exatamente os placeholders `{query}` e `{context}` do prompt

**StrOutputParser**:
- LLMs retornam objetos `AIMessage` — não strings simples
- `StrOutputParser()` extrai apenas o `.content` do `AIMessage`, retornando uma string limpa

In [28]:
setup_retrival = RunnableParallel({
    'query': RunnablePassthrough(),
    'context': parent_document_retriever,
})

output_parser = StrOutputParser()

### Chain Final com Operador Pipe `|`

O operador `|` é a sintaxe LCEL para encadear runnables em sequência:

```
setup_retrival | rag_prompt | llm | output_parser
```

Cada etapa recebe a saída da anterior:
1. `setup_retrival` → recebe a pergunta, retorna `{query, context}`
2. `rag_prompt` → recebe o dicionário, retorna o prompt formatado
3. `llm` → recebe o prompt, retorna `AIMessage` com a resposta
4. `output_parser` → recebe o `AIMessage`, retorna string limpa

Esta notação é equivalente a chamadas aninhadas, mas muito mais legível

In [29]:
parent_chain_retrival = setup_retrival | rag_prompt | llm | output_parser

In [31]:
from IPython.display import display, Markdown

question = 'Quais os principais riscos do marco legal de IA?'
answer = parent_chain_retrival.invoke(question)

display(Markdown(f"""
---
**Pergunta:** {question}

**Resposta:**

{answer}

---
"""))


---
**Pergunta:** Quais os principais riscos do marco legal de IA?

**Resposta:**

Alguns dos principais riscos do marco legal de IA podem incluir:

1. Violação dos direitos fundamentais: Se o marco legal de IA não conseguir proteger adequadamente os direitos fundamentais das pessoas, poderá haver discriminação, violação da privacidade e outras formas de injustiça.

2. Falta de segurança e confiabilidade dos sistemas de IA: Caso as normas estabelecidas não garantam a implementação de sistemas de IA seguros e confiáveis, podem ocorrer acidentes e danos irreparáveis.

3. Impactos no desenvolvimento científico e tecnológico: Se as regulamentações forem muito restritivas, isso pode limitar a inovação e o progresso tecnológico no campo da Inteligência Artificial.

4. Falta de transparência e responsabilidade: Se o marco legal não exigir transparência nos algoritmos e nas decisões tomadas pelos sistemas de IA, isso pode dificultar a responsabilização em caso de erros ou danos causados pelas máquinas.

Esses são apenas alguns dos possíveis riscos associados ao marco legal de IA, sendo essencial que a legislação seja abrangente e equilibrada para promover o uso responsável e ético da inteligência artificial.

---
